# Fabric Table Preview with PySpark

command to run the jupyter server for this notebook: 

`runjupyter (in any folder)`

equivalent to running this in /go-api:

```docker compose run --rm -p 8888:8888 serve jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root --notebook-dir=/home/ifrc```


In [17]:
import os
from pathlib import Path
from urllib.request import urlretrieve

POSTGRES_JDBC_VERSION = '42.7.4'
POSTGRES_JDBC_JAR = Path(f'/tmp/postgresql-{POSTGRES_JDBC_VERSION}.jar')
POSTGRES_JDBC_URL = (
    'https://repo1.maven.org/maven2/org/postgresql/postgresql/'
    f'{POSTGRES_JDBC_VERSION}/postgresql-{POSTGRES_JDBC_VERSION}.jar'
)

if not POSTGRES_JDBC_JAR.exists():
    urlretrieve(POSTGRES_JDBC_URL, POSTGRES_JDBC_JAR)

# Must be set before Spark JVM starts in this kernel
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--jars {POSTGRES_JDBC_JAR} pyspark-shell'

from pyspark.sql import SparkSession

In [18]:
# Stop the current Spark session so this cell can be rerun safely
active_spark = SparkSession.getActiveSession()
if active_spark is not None:
    active_spark.stop()

SPARK_MASTER = os.getenv("SPARK_MASTER", "local[*]")  # go to .env in go-api to configure, default to local 

spark = (
    SparkSession.builder
    .appName('postgres-stock-inventory')
    .master(SPARK_MASTER)
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

spark

In [19]:
# These are provided by docker compose env + .env when running notebook in `serve` container
PG_HOST = os.getenv('DJANGO_DB_HOST', 'db')
PG_PORT = os.getenv('DJANGO_DB_PORT', '5432')
PG_DB = os.environ['DJANGO_DB_NAME']
PG_USER = os.environ['DJANGO_DB_USER']
PG_PASSWORD = os.environ['DJANGO_DB_PASS']

jdbc_url = f'jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}'

# Read only table names in public schema to discover what you can load
table_list_query = """(
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name
) t"""

tables_df = (
    spark.read.format('jdbc')
    .option('url', jdbc_url)
    .option('dbtable', table_list_query)
    .option('user', PG_USER)
    .option('password', PG_PASSWORD)
    .option('driver', 'org.postgresql.Driver')
    .load()
)

print('Total public tables:', tables_df.count())
tables_df.show(5, truncate=False)

Total public tables: 352
+------------------------+
|table_name              |
+------------------------+
|api_action              |
|api_actionstaken        |
|api_actionstaken_actions|
|api_admin2              |
|api_admin2geoms         |
+------------------------+
only showing top 5 rows



In [20]:
# Load all required tables from Postgres as DataFrames
print("Loading tables from Postgres...")

dimwarehouse_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimwarehouse")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproduct_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproduct")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransactionline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransactionline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventorytransaction_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventorytransaction")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryowner_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryowner")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductcategory_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductcategory")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitem_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitem")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

diminventoryitemstatus_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_diminventoryitemstatus")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

dimproductreceiptline_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_dimproductreceiptline")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

print("✓ All tables loaded successfully")
print(f"  - dimwarehouse: {dimwarehouse_df.count()} rows")
print(f"  - dimproduct: {dimproduct_df.count()} rows")
print(f"  - diminventorytransactionline: {diminventorytransactionline_df.count()} rows")
print(f"  - diminventorytransaction: {diminventorytransaction_df.count()} rows")
print(f"  - diminventoryowner: {diminventoryowner_df.count()} rows")
print(f"  - dimproductcategory: {dimproductcategory_df.count()} rows")
print(f"  - diminventoryitem: {diminventoryitem_df.count()} rows")
print(f"  - diminventoryitemstatus: {diminventoryitemstatus_df.count()} rows")
print(f"  - dimproductreceiptline: {dimproductreceiptline_df.count()} rows")


Loading tables from Postgres...
✓ All tables loaded successfully
  - dimwarehouse: 1172 rows
  - dimproduct: 27640 rows
  - diminventorytransactionline: 24799 rows
  - diminventorytransaction: 21636 rows
  - diminventoryowner: 109 rows
  - dimproductcategory: 3273 rows
  - diminventoryitem: 27506 rows
  - diminventoryitemstatus: 6 rows
  - dimproductreceiptline: 15649 rows


In [21]:
# Register all DataFrames as SQL temp views
dimwarehouse_df.createOrReplaceTempView("dimwarehouse")
dimproduct_df.createOrReplaceTempView("dimproduct")
dimproductcategory_df.createOrReplaceTempView("dimproductcategory")
diminventorytransactionline_df.createOrReplaceTempView("diminventorytransactionline")
diminventorytransaction_df.createOrReplaceTempView("diminventorytransaction")
diminventoryowner_df.createOrReplaceTempView("diminventoryowner")
diminventoryitem_df.createOrReplaceTempView("diminventoryitem")
diminventoryitemstatus_df.createOrReplaceTempView("diminventoryitemstatus")
dimproductreceiptline_df.createOrReplaceTempView("dimproductreceiptline")

print("✓ All tables registered as SQL views")


✓ All tables registered as SQL views


In [22]:
#filter diminventorytransaction before joining

df = spark.sql("""
    select 
        *
    from diminventorytransaction
    where 
        reference_category NOT ILIKE '%Weighted average inventory closing%'
        and NOT excluded_from_inventory_value
    order by id
"""
)


df.createOrReplaceTempView("diminventorytransaction")

spark.sql("""
    select 
        *
    from diminventorytransaction
    where 
        reference_category ILIKE '%Weighted average inventory closing%'
""").show()


+---+------------------+----------------+-----------------------------+
| id|reference_category|reference_number|excluded_from_inventory_value|
+---+------------------+----------------+-----------------------------+
+---+------------------+----------------+-----------------------------+



In [23]:
#filter warehouses down to only these specific warehouse codes


df = spark.sql("""
    select
        *
    from dimwarehouse
    WHERE id IN (
    'AE1DUB002',
    'AR1BUE002',
    'AU1BRI003',
    'ES1LAS001',
    'GT1GUA001',
    'HN1COM002',
    'MY1SEL001',
    'PA1ARR001',
    'TR1ISTA02'
)
""")

df.show()

df.createOrReplaceTempView("dimwarehouse")

+---------+----+--------------------+--------------------+-------+
|       id|site|                name|      postal_address|country|
+---------+----+--------------------+--------------------+-------+
|AE1DUB002| AE1|IFRC Reg - Dubai ...|International Hum...|    ARE|
|AR1BUE002| PA1|IFRC Sub-Reg - Ar...|Aeropuerto Intern...|    ARG|
|AU1BRI003| MY1|IFRC Sub-Reg - Br...|113 Bancroft Road...|    AUS|
|ES1LAS001| AE1|IFRC Sub-Reg - La...|IFRC - CENTRO LOG...|    ESP|
|GT1GUA001| PA1|IFRC Sub-Reg - Gu...|FEDERACIÓN INTERN...|    GTM|
|HN1COM002| PA1|IFRC Sub-Reg - Ho...|Anillo Periferico...|    HND|
|MY1SEL001| MY1|IFRC Reg - Malays...|INTEGRATED LOGIST...|    MYS|
|PA1ARR001| PA1|IFRC Reg - Panama...|Regional Logistic...|    PAN|
|TR1ISTA02| AE1|Kizilay Logistics...|Akfirat, Kizilay,...|    TUR|
+---------+----+--------------------+--------------------+-------+



In [24]:
#filter diminventorytransactionline 
#transactions where the owner code is IFRC
#specific status
#status must be OK
#must not be returned

df = spark.sql("""
    select * 
    from diminventorytransactionline
    where 
        owner not ilike '%#ifrc%'
        and status IN (
            'Deducted',
            'Purchased',
            'Received',
            'Reserved physical',
            'Sold'
            )
        and item_status = 'OK'
        and NOT packing_slip_returned
""")

df.createOrReplaceTempView("diminventorytransactionline")
#more specifically the #... within id must not be #ifrc


In [25]:
spark.sql("""
    select * from dimproduct where unit_of_measure = 'EA'
""").show(10)

+--------------+--------------------+----+---------------+----------------+--------------------+
|            id|                name|type|unit_of_measure|product_category|    project_category|
+--------------+--------------------+----+---------------+----------------+--------------------+
|ASOUACCSZ00058|          Travel Mug|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUACCSZ00059|              Bottle|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUACCSZ00060|                Lamp|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUACCSZ00061|              Folder|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUACCSZ00062|        Moleskin set|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUACCSZ00063|        Moleskin red|Item|             EA|        ASOUACCS|IFRC#7908I - COST...|
|ASOUFLAGZ00080|Flag Crystal 200*...|Item|             EA|        ASOUFLAG|IFRC#7908I - COST...|
|ASOUFLAGZ00081|Flag Crystal 1

In [26]:
#filter out categories that are 'services'

df = spark.sql("""
    select * from dimproductcategory where name not ilike 'services'
""")

df.createOrReplaceTempView("dimproductcategory")

spark.sql("""
    select * from dimproductcategory where name ilike 'services'
""").show()

+-------------+----+--------------------+-----+
|category_code|name|parent_category_code|level|
+-------------+----+--------------------+-----+
+-------------+----+--------------------+-----+



In [27]:
df = spark.sql("""
    select
        w.id as warehouse_id
        ,w.name as warehouse
        ,w.country as warehouse_country
        ,pc.name as product_category
        ,p.name as item_name
        ,CAST(ROUND(SUM(quantity), 2) AS DECIMAL(18,2)) AS quantity
        ,lower(p.unit_of_measure) as unit_measurement
    from dimwarehouse w
    join diminventorytransactionline itl
        on w.id = itl.warehouse
    join diminventorytransaction it
        on itl.inventory_transaction = it.id
    join dimproduct p
        on itl.product = p.id
    join dimproductcategory pc
        on p.product_category = pc.category_code
    group by w.name, w.id, w.country, pc.name, p.name, lower(p.unit_of_measure)
    having quantity > 0
    order by warehouse, product_category, quantity desc
    
""")


In [28]:
#write to the stockinventory table in database

df.write.format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "api_stockinventory") \
    .option("user", PG_USER) \
    .option("password", PG_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite")  \
    .save()

the below is for local dev purposes so remove for production

In [29]:
stockinventory_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "api_stockinventory")
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
)

stockinventory_df.createOrReplaceTempView("stockinventory")

spark.sql("""
    select * from stockinventory
""").show()

+------------+--------------------+-----------------+--------------------+--------------------+---------+----------------+
|warehouse_id|           warehouse|warehouse_country|    product_category|           item_name| quantity|unit_measurement|
+------------+--------------------+-----------------+--------------------+--------------------+---------+----------------+
|   AE1DUB002|IFRC Reg - Dubai ...|              ARE|            Blankets|BLANKET, SYNTHETI...|   180.00|              ea|
|   AE1DUB002|IFRC Reg - Dubai ...|              ARE|Buckets (Containers)|BUCKET, plastic, ...|  3560.00|              ea|
|   AE1DUB002|IFRC Reg - Dubai ...|              ARE|Buckets (Containers)|BUCKET, plastic, ...|  1400.00|              ea|
|   AE1DUB002|IFRC Reg - Dubai ...|              ARE|              Chlore|CHLORINE,40mg (Na...|208000.00|              ea|
|   AE1DUB002|IFRC Reg - Dubai ...|              ARE|              Chlore|CHLORINE, NaDCC P...|    20.00|              ea|
|   AE1DUB002|IF

In [30]:
# stockinventory_df.toPandas().to_csv("go-api/scripts/pyspark/stock_inventory.csv", index=False)

In [40]:
import importlib
import os
import sys
from pathlib import Path

# Prioritize the current workspace path so notebook imports your edited code.
preferred_repo = "/home/huanghaogao/systems_engineering/go-web-app-ucl/go-api"
if Path(preferred_repo).exists():
    if preferred_repo in sys.path:
        sys.path.remove(preferred_repo)
    sys.path.insert(0, preferred_repo)

# Keep the container path as fallback only.
fallback_repo = "/home/ifrc/go-api"
if Path(fallback_repo).exists() and fallback_repo not in sys.path:
    sys.path.append(fallback_repo)

# Required because the imported module depends on Django models/settings.
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "main.settings")

import django
django.setup()

import api.data_transformation_framework_agreement as dtfa
importlib.reload(dtfa)

get_country_region_mapping = dtfa.get_country_region_mapping

mapping_df = get_country_region_mapping(spark)
mapping_df.select("iso2", "iso3", "country_name", "region").show(20, truncate=False)

+----+----+-------------------+------------+
|iso2|iso3|country_name       |region      |
+----+----+-------------------+------------+
|AF  |AFG |Afghanistan        |Asia-Pacific|
|AL  |ALB |Albania            |Europe      |
|DZ  |DZA |Algeria            |MENA        |
|AS  |ASM |American Samoa     |Americas    |
|AD  |AND |Andorra            |Europe      |
|AO  |AGO |Angola             |Africa      |
|AI  |AIA |Anguilla           |            |
|AQ  |ATA |Antarctica         |            |
|AG  |ATG |Antigua and Barbuda|Americas    |
|AR  |ARG |Argentina          |Americas    |
|AM  |ARM |Armenia            |Europe      |
|AW  |ABW |Aruba              |Americas    |
|AU  |AUS |Australia          |Asia-Pacific|
|AT  |AUT |Austria            |Europe      |
|AZ  |AZE |Azerbaijan         |Europe      |
|BS  |BHS |Bahamas            |Americas    |
|BH  |BHR |Bahrain            |MENA        |
|BD  |BGD |Bangladesh         |Asia-Pacific|
|BB  |BRB |Barbados           |Americas    |
|BY  |BLR 